# Attempt to get Liquidation count and liquidation notional values from Binance

In [30]:
import asyncio
import json
import os
from websockets import connect

websocket_uri = "wss://fstream.binance.com/ws/btcusdt@forceOrder" # whats does uri mean?
filename = "binance_btc.csv" # saves data as a csv file

if not os.path.isfile(filename): # checks if file already exists
    with open(filename, "w") as f:  # how does local variable f be accessible outside of this block? # the filename can be accessed and f can be used again to write

        f.write(",".join(["symbol","side","order_type","time_in_force",
                          "original_quantity","price","average_price",
                          "order_status","order_last_filled_quantity",
                          "order_filled_accumalated_quantity",
                          "order_trade_time"]) + "\n")   # sorting data from binance in this format and columns are seperated by commas


async def binance_liquidations(uri, filename): # defining an asynchronous function, meaning if one part is waiting, other parts can run ex. liquidation for BTCUSDT is waiting, liquidation for ETHUSDT can run
    async for websocket in connect(uri): # connects to the websocket uri
        try:
            while True: # infinite loop why?
                msg = await websocket.recv()
                print(msg) # prints the message received from the websocket
                msg = json.loads(msg)["o"] # doing json.loads making the json string into a python dictionary, then accessing the "o" key which contains the order data
                msg = [ str(x) for x in list(msg.values())] # converting all values in the dictionary to strings and creating a List of these values Genius!!
                with open(filename, "a") as f: # opening the file again in append mode
                    f.write(",".join(msg) + "\n") # writing the message to the file, joining the list into a comma seperated string eg. ["a","b","c"] -> 'a,b,c'\n
        except Exception as e: 
            print(e) # prints any exceptions that occur
            continue # retry the connection as long as possible

asyncio.run(await binance_liquidations(websocket_uri, filename))

# For jupyter use: await binance_liquidations( ... )

: 

: 

In [24]:
import pandas as pd
data = pd.read_csv("binance.csv")
data

,symbol,side,order_type,time_in_force,original_quantity,price,average_price,order_status,order_last_filled_quantity,order_filled_accumalated_quantity,order_trade_time
0,ORDERUSDT,SELL,LIMIT,IOC,236.0,0.280540,0.284651,FILLED,210.0,236.0,1758947582121
1,IOSTUSDT,SELL,LIMIT,IOC,3697.0,0.002966,0.003009,FILLED,3697.0,3697.0,1758947592046
2,ALPINEUSDT,SELL,LIMIT,IOC,20.0,5.111900,5.185700,FILLED,20.0,20.0,1758947592444
3,COAIUSDT,SELL,LIMIT,IOC,7000.0,0.151120,0.153596,FILLED,1607.0,7000.0,1758947605299
4,ALPINEUSDT,SELL,LIMIT,IOC,1.0,5.105900,5.182600,FILLED,1.0,1.0,1758947605542
...,...,...,...,...,...,...,...,...,...,...,...
212,LIGHTUSDT,SELL,LIMIT,IOC,59.0,0.964900,0.977610,FILLED,59.0,59.0,1759087578146
213,HIFIUSDT,BUY,LIMIT,IOC,10311.0,0.148520,0.146101,FILLED,353.0,10311.0,1759087579436
214,LIGHTUSDT,SELL,LIMIT,IOC,19.0,0.963255,0.981061,FILLED,11.0,19.0,1759087580933
215,LIGHTUSDT,SELL,LIMIT,IOC,45.0,0.960210,0.970810,FILLED,45.0,45.0,1759087590319


In [25]:
data["order time"] = pd.to_datetime(data["order_trade_time"], unit='ms', utc=True).dt.tz_convert('US/Pacific')
data

,symbol,side,order_type,time_in_force,original_quantity,price,average_price,order_status,order_last_filled_quantity,order_filled_accumalated_quantity,order_trade_time,order time
0,ORDERUSDT,SELL,LIMIT,IOC,236.0,0.280540,0.284651,FILLED,210.0,236.0,1758947582121,2025-09-26 21:33:02.121000-07:00
1,IOSTUSDT,SELL,LIMIT,IOC,3697.0,0.002966,0.003009,FILLED,3697.0,3697.0,1758947592046,2025-09-26 21:33:12.046000-07:00
2,ALPINEUSDT,SELL,LIMIT,IOC,20.0,5.111900,5.185700,FILLED,20.0,20.0,1758947592444,2025-09-26 21:33:12.444000-07:00
3,COAIUSDT,SELL,LIMIT,IOC,7000.0,0.151120,0.153596,FILLED,1607.0,7000.0,1758947605299,2025-09-26 21:33:25.299000-07:00
4,ALPINEUSDT,SELL,LIMIT,IOC,1.0,5.105900,5.182600,FILLED,1.0,1.0,1758947605542,2025-09-26 21:33:25.542000-07:00
...,...,...,...,...,...,...,...,...,...,...,...,...
212,LIGHTUSDT,SELL,LIMIT,IOC,59.0,0.964900,0.977610,FILLED,59.0,59.0,1759087578146,2025-09-28 12:26:18.146000-07:00
213,HIFIUSDT,BUY,LIMIT,IOC,10311.0,0.148520,0.146101,FILLED,353.0,10311.0,1759087579436,2025-09-28 12:26:19.436000-07:00
214,LIGHTUSDT,SELL,LIMIT,IOC,19.0,0.963255,0.981061,FILLED,11.0,19.0,1759087580933,2025-09-28 12:26:20.933000-07:00
215,LIGHTUSDT,SELL,LIMIT,IOC,45.0,0.960210,0.970810,FILLED,45.0,45.0,1759087590319,2025-09-28 12:26:30.319000-07:00


In [26]:
data.drop(columns=["order_type"], inplace=True)
data.drop(columns=["original_quantity"], inplace=True)
data.drop(columns=["price"], inplace=True)
data.drop(columns=["order_status"], inplace=True)
data.drop(columns=["order_last_filled_quantity"], inplace=True)

In [27]:
data['notional'] = data['average_price'].astype(float) * data['order_filled_accumalated_quantity'].astype(float)
data

,symbol,side,time_in_force,average_price,order_filled_accumalated_quantity,order_trade_time,order time,notional
0,ORDERUSDT,SELL,IOC,0.284651,236.0,1758947582121,2025-09-26 21:33:02.121000-07:00,67.177636
1,IOSTUSDT,SELL,IOC,0.003009,3697.0,1758947592046,2025-09-26 21:33:12.046000-07:00,11.124273
2,ALPINEUSDT,SELL,IOC,5.185700,20.0,1758947592444,2025-09-26 21:33:12.444000-07:00,103.714000
3,COAIUSDT,SELL,IOC,0.153596,7000.0,1758947605299,2025-09-26 21:33:25.299000-07:00,1075.171300
4,ALPINEUSDT,SELL,IOC,5.182600,1.0,1758947605542,2025-09-26 21:33:25.542000-07:00,5.182600
...,...,...,...,...,...,...,...,...
212,LIGHTUSDT,SELL,IOC,0.977610,59.0,1759087578146,2025-09-28 12:26:18.146000-07:00,57.678990
213,HIFIUSDT,BUY,IOC,0.146101,10311.0,1759087579436,2025-09-28 12:26:19.436000-07:00,1506.446380
214,LIGHTUSDT,SELL,IOC,0.981061,19.0,1759087580933,2025-09-28 12:26:20.933000-07:00,18.640149
215,LIGHTUSDT,SELL,IOC,0.970810,45.0,1759087590319,2025-09-28 12:26:30.319000-07:00,43.686450


In [28]:
#identify all crypto currenciens that got liquidated
data['symbol'].unique().tolist()

['ORDERUSDT',
 'IOSTUSDT',
 'ALPINEUSDT',
 'COAIUSDT',
 'HUSDT',
 'ASRUSDT',
 'XPLUSDT',
 'BLESSUSDT',
 'DAMUSDT',
 'OPENUSDT',
 'XPINUSDT',
 'SNXUSDT',
 'BARDUSDT',
 'IPUSDT',
 'MYXUSDT',
 'AVNTUSDT',
 'MIRAUSDT',
 'STBLUSDT',
 'HEMIUSDT',
 'SQDUSDT',
 'AKEUSDT',
 'FLUIDUSDT',
 'DOODUSDT',
 'ASTERUSDT',
 'LINEAUSDT',
 'WUSDT',
 'KMNOUSDT',
 'ALCHUSDT',
 'DEGOUSDT',
 'HANAUSDT',
 'HIFIUSDT',
 'FORMUSDT',
 'ICNTUSDT',
 'THEUSDT',
 'KAITOUSDT',
 'LIGHTUSDT',
 'MUSDT',
 'IDOLUSDT',
 'BUSDT',
 'ZORAUSDT',
 'DENTUSDT',
 'SUIUSDT',
 'UBUSDT',
 'SKYUSDT',
 'XNYUSDT',
 '0GUSDT',
 'DMCUSDT',
 'TAUSDT',
 'BERAUSDT',
 'FLOCKUSDT',
 'WIFUSDT']

In [29]:
# identify most frequently liquidated crypto
data['symbol'].value_counts()

symbol
AKEUSDT       23
LIGHTUSDT     18
HIFIUSDT      15
ICNTUSDT      14
FORMUSDT      12
XPLUSDT       10
SQDUSDT        9
SNXUSDT        9
ALPINEUSDT     9
DOODUSDT       8
BLESSUSDT      7
ORDERUSDT      7
ASTERUSDT      6
BARDUSDT       6
BERAUSDT       6
MYXUSDT        5
AVNTUSDT       4
UBUSDT         4
THEUSDT        4
HANAUSDT       3
COAIUSDT       2
MIRAUSDT       2
HUSDT          2
0GUSDT         2
KAITOUSDT      2
WUSDT          2
FLUIDUSDT      2
HEMIUSDT       1
STBLUSDT       1
IPUSDT         1
XPINUSDT       1
ASRUSDT        1
DAMUSDT        1
OPENUSDT       1
IOSTUSDT       1
DEGOUSDT       1
ALCHUSDT       1
LINEAUSDT      1
KMNOUSDT       1
BUSDT          1
ZORAUSDT       1
IDOLUSDT       1
MUSDT          1
SUIUSDT        1
DENTUSDT       1
XNYUSDT        1
SKYUSDT        1
DMCUSDT        1
TAUSDT         1
FLOCKUSDT      1
WIFUSDT        1
Name: count, dtype: int64